# Colab Training Notebook

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q torchmetrics mlflow tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 26.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 119.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import torch
import shutil, os, glob
import mlflow
import pandas as pd
import sys
from dotenv import load_dotenv
from pathlib import Path

In [4]:
load_dotenv()

True

In [5]:
# Check GPU
if torch.cuda.is_available():
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [6]:
# Set Drive path
DRIVE_PATH = Path(os.environ["DRIVE_PATH"])
os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/data/splits', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/data/labeled', exist_ok=True)
print("Drive path ready")

Drive path ready


In [7]:
if str(DRIVE_PATH) not in sys.path:
    sys.path.append(str(DRIVE_PATH))

from src.train import train_model
from src.evaluate import evaluate 

In [8]:
# Set MLflow tracking URI to local SQLite database
mlflow.set_tracking_uri(f"sqlite:///{DRIVE_PATH}/mlflow.db")
mlflow.set_experiment("visiomark-image-model")
print("MLflow ready")

MLflow ready


In [9]:
# Check data splits
splits = {}
for split in ['train', 'val', 'test']:
    path = f"{DRIVE_PATH}/data/splits/{split}.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        splits[split] = df
        print(f"{split}: {len(df)} samples")
        print(f"content_type distribution: {df['content_type_label'].value_counts().to_dict()}")
        print(f"mood distribution:         {df['mood_label'].value_counts().to_dict()}")
        print()
    else:
        print(f"{split}.csv not found")

train: 1131 samples
content_type distribution: {1: 609, 2: 355, 0: 167}
mood distribution:         {1: 423, 2: 352, 0: 224, 3: 132}

val: 243 samples
content_type distribution: {1: 131, 2: 76, 0: 36}
mood distribution:         {1: 91, 2: 76, 0: 47, 3: 29}

test: 243 samples
content_type distribution: {1: 131, 2: 76, 0: 36}
mood distribution:         {1: 90, 2: 76, 0: 48, 3: 29}



### Phase 1

In [10]:
print("Starting Phase 1.")
print("=" * 60)

checkpoints_path = train_model(
    phase=1,
    epochs=15,
    batch_size=16,
    lr=1e-3,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    train_csv=f"{DRIVE_PATH}/data/splits/train.csv",
    val_csv=f"{DRIVE_PATH}/data/splits/val.csv",
    checkpoint=None,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_dir=f"{DRIVE_PATH}/checkpoints",
)

print("Phase 1 checkpoints saved to: /checkpoints ")

Starting Phase 1.
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 52.6MB/s]


Epoch 01/15 | loss=2.3404 | F1_content=0.5959 | F1_mood=0.4777 | Avg_F1=0.5368


2026/06/08 15:41:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 15:41:39 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/08 15:41:40 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/08 15:41:52 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version

    Checkpoint saved (avg_F1=0.5368)


...

    Checkpoint saved (avg_F1=0.6150)
Epoch 08/15 | loss=1.6897 | F1_content=0.6553 | F1_mood=0.5547 | Avg_F1=0.6050
Epoch 09/15 | loss=1.6793 | F1_content=0.6550 | F1_mood=0.5568 | Avg_F1=0.6059


2026/06/08 15:46:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 10/15 | loss=1.6748 | F1_content=0.6539 | F1_mood=0.5836 | Avg_F1=0.6188


2026/06/08 15:46:04 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/08 15:46:04 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/08 15:46:12 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the loca

    Checkpoint saved (avg_F1=0.6188)
Epoch 11/15 | loss=1.5991 | F1_content=0.6387 | F1_mood=0.5886 | Avg_F1=0.6137
Epoch 12/15 | loss=1.6256 | F1_content=0.6522 | F1_mood=0.5434 | Avg_F1=0.5978
Epoch 13/15 | loss=1.6212 | F1_content=0.6418 | F1_mood=0.5563 | Avg_F1=0.5991
Epoch 14/15 | loss=1.6049 | F1_content=0.6513 | F1_mood=0.5834 | Avg_F1=0.6173
Epoch 15/15 | loss=1.6351 | F1_content=0.6468 | F1_mood=0.5692 | Avg_F1=0.6080

Phase 1 done. Best avg F1 = 0.6188
Phase 1 checkpoints saved to: /checkpoints 


### Phase 2

In [11]:
print("Starting Phase 2.")
print("=" * 60)

phase1_checkpoints = sorted(glob.glob(f'{DRIVE_PATH}/checkpoints/best_phase1*.pt'))

if not phase1_checkpoints:
    raise FileNotFoundError("No Phase 1 checkpoint found.")

best_phase1 = phase1_checkpoints[-1]

final_checkpoints_path = train_model(
    phase=2,
    epochs=50,
    batch_size=16,
    lr=1e-5,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    train_csv=f"{DRIVE_PATH}/data/splits/train.csv",
    val_csv=f"{DRIVE_PATH}/data/splits/val.csv",
    checkpoint=best_phase1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_dir=f"{DRIVE_PATH}/checkpoints",
)

print("Phase 2 checkpoints saved to: /checkpoints ")

Starting Phase 2.
  Loading Phase 1 checkpoint
  Unfroze last 3 encoder blocks
Epoch 01/50 | loss=1.6260 | F1_content=0.6499 | F1_mood=0.5808 | Avg_F1=0.6153


2026/06/08 15:48:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/08 15:48:43 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/08 15:48:43 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/08 15:48:49 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version

    Checkpoint saved (avg_F1=0.6153)


2026/06/08 15:49:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 02/50 | loss=1.5783 | F1_content=0.6665 | F1_mood=0.5724 | Avg_F1=0.6194


...

    Checkpoint saved (avg_F1=0.6777)
Epoch 36/50 | loss=1.0297 | F1_content=0.6989 | F1_mood=0.6457 | Avg_F1=0.6723
Epoch 37/50 | loss=1.0327 | F1_content=0.6929 | F1_mood=0.6417 | Avg_F1=0.6673
Epoch 38/50 | loss=1.0534 | F1_content=0.6974 | F1_mood=0.6265 | Avg_F1=0.6620
Epoch 39/50 | loss=1.0408 | F1_content=0.7032 | F1_mood=0.6427 | Avg_F1=0.6729
Epoch 40/50 | loss=1.0650 | F1_content=0.7066 | F1_mood=0.6381 | Avg_F1=0.6724
Epoch 41/50 | loss=1.0683 | F1_content=0.6973 | F1_mood=0.6322 | Avg_F1=0.6647


2026/06/08 16:08:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 42/50 | loss=0.9826 | F1_content=0.7074 | F1_mood=0.6625 | Avg_F1=0.6850


2026/06/08 16:08:51 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/08 16:08:52 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/08 16:08:59 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the loca

    Checkpoint saved (avg_F1=0.6850)
Epoch 43/50 | loss=1.0627 | F1_content=0.7148 | F1_mood=0.6492 | Avg_F1=0.6820
Epoch 44/50 | loss=1.0507 | F1_content=0.7039 | F1_mood=0.6372 | Avg_F1=0.6706
Epoch 45/50 | loss=1.0326 | F1_content=0.7087 | F1_mood=0.6538 | Avg_F1=0.6813
Epoch 46/50 | loss=1.0206 | F1_content=0.6999 | F1_mood=0.6476 | Avg_F1=0.6737
Epoch 47/50 | loss=1.0229 | F1_content=0.7150 | F1_mood=0.6495 | Avg_F1=0.6823
Epoch 48/50 | loss=1.0400 | F1_content=0.7131 | F1_mood=0.6435 | Avg_F1=0.6783
Epoch 49/50 | loss=1.0357 | F1_content=0.7129 | F1_mood=0.6524 | Avg_F1=0.6826
Epoch 50/50 | loss=1.0325 | F1_content=0.7025 | F1_mood=0.6571 | Avg_F1=0.6798

Phase 2 done. Best avg F1 = 0.6850
Phase 2 checkpoints saved to: /checkpoints 


In [12]:
# Copy MLflow DB if using local mode
db_source = f"{DRIVE_PATH}/mlflow.db"
db_backup = f"{DRIVE_PATH}/mlflow_backup.db"

if os.path.exists(db_source):
    shutil.copy2(db_source, db_backup)
    print("MLflow database backed up saved.")
else:
    print("MLflow DB not found.")

print("\nALL TRAINING COMPLETE!")

MLflow database backed up saved.

ALL TRAINING COMPLETE!


## Evaluation

In [13]:
phase2_checkpoints = sorted(glob.glob(f'{DRIVE_PATH}/checkpoints/best_phase2*.pt'))

In [14]:
checkpoint_path = phase2_checkpoints[-1] if phase2_checkpoints else None
if checkpoint_path is None:
    raise FileNotFoundError("No Phase 2 checkpoint found for evaluation.")
print("Checkpoint ready for evaluation")

Checkpoint ready for evaluation


In [15]:
result = evaluate(
    checkpoint=checkpoint_path,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    test_csv=f"{DRIVE_PATH}/data/splits/test.csv",
    save_path=f"{DRIVE_PATH}/eval/final_results.json",
)
print("Results saved to eval/final_results.json")

Loading model on cuda
Loading test data

TEST RESULTS
Content Type F1 : 0.6987
Mood F1         : 0.5749
Average F1      : 0.6368

Confusion Matrices

Content Type Confusion Matrix:

Mood Confusion Matrix:
Saved 'eval/content_type_confusion_matrix.png' and 'eval/mood_confusion_matrix.png'

Content Type Classification Report:
                  precision    recall  f1-score   support

Product Showcase       0.43      0.61      0.51        36
       Lifestyle       0.79      0.70      0.74       131
     Promotional       0.71      0.71      0.71        76

        accuracy                           0.69       243
       macro avg       0.65      0.67      0.65       243
    weighted avg       0.71      0.69      0.70       243


Mood Classification Report:
              precision    recall  f1-score   support

   Energetic       0.48      0.46      0.47        48
        Calm       0.64      0.57      0.60        90
Professional       0.70      0.64      0.67        76
     Playful       

#### Extract Image Vectors for Alignment Training


In [7]:
!pip install -q torchmetrics transformers tqdm

In [12]:
# Extract image vectors from alignment pairs
import os
import torch
import pandas as pd
from PIL import Image
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load model
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).to(DEVICE)
model.classifier = torch.nn.Identity() 
model.eval()

# Load alignment pairs CSV
# Expected columns: image_name, caption
ALIGNMENT_CSV = f'{DRIVE_PATH}/data/alignment_pairs/alignment_pairs.csv'

if not os.path.exists(ALIGNMENT_CSV):
    print("alignment_pairs.csv not found.")
    print("Upload data/alignment_pairs/ folder first.")
else:
    df = pd.read_csv(ALIGNMENT_CSV)
    print(f"Loaded {len(df)} alignment pairs")

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    image_vectors = []
    valid_idx = []
    cleaned_captions = []
    
    print("Extracting vectors for manual pairs")

    with torch.no_grad():
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
            caption = str(row.get("caption", "")).strip()
            words = caption.split()
            word_count = len(words)
            
            if word_count < 8:
                continue  
            
            if word_count > 25:
                words = words[:25]
                caption = " ".join(words)
            
            img_path = f'{DRIVE_PATH}/data/raw/{row["image_name"]}'
            try:
                img = Image.open(img_path).convert('RGB')
                img_t = transform(img).unsqueeze(0).to(DEVICE)
                vec = model(img_t).squeeze(0).cpu()
                
                image_vectors.append(vec)
                valid_idx.append(idx)
                cleaned_captions.append(caption)
            except Exception as e:
                print(f"  Skip {row['img_path']}: {e}")

    image_vectors = torch.stack(image_vectors)  # [N, 1280]
    df_valid = df.iloc[valid_idx].reset_index(drop=True)
    
    df_valid["caption"] = cleaned_captions

    # Save
    torch.save(image_vectors, f'{DRIVE_PATH}/checkpoints/alignment_image_vectors.pt')
    df_valid.to_csv(f'{DRIVE_PATH}/data/alignment_pairs/alignment_pairs_valid.csv', index=False)

    print(f"\n {len(image_vectors)} image vectors saved → Drive")
    print(f"   Shape: {image_vectors.shape}")
    print(f"   Filtered out: {len(df) - len(df_valid)} rows")

Loaded 201 alignment pairs
Extracting vectors for manual pairs


Extracting: 100%|██████████| 201/201 [00:03<00:00, 55.82it/s]


 193 image vectors saved → Drive
   Shape: torch.Size([193, 1280])
   Filtered out: 8 rows
